# CardingForums — 03. Análisis semántico

En este notebook usamos **inteligencia artificial** para entender el *contenido* de lo que se dice en los foros, no solo *quién* habla con *quién*.

Mientras el análisis estructural (notebook 02) trabaja con la red de relaciones, el análisis semántico trabaja con el texto en sí:
- ¿De qué habla cada usuario?
- ¿Qué temas dominan el foro?
- ¿Podemos inferir el ROL de cada actor a partir de su vocabulario?

**Requisito previo**: este notebook requiere [Ollama](https://ollama.com/) corriendo localmente con los modelos:
- `qwen3-embedding` — para generar embeddings de texto
- `qwen2.5:14b` — para NER y perfilado de actores

```bash
ollama pull qwen3-embedding
ollama pull qwen2.5:14b
```

---

## 1. Setup

Cargamos los datos procesados del notebook 01 y las librerías necesarias.

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
import plotly.express as px
import json
from pathlib import Path
from tqdm.notebook import tqdm

from src.utils import RESULTS_DIR
from src.embeddings import embed_users, cosine_similarity

# Datos del notebook 01
users  = pd.read_parquet(RESULTS_DIR / "01_users_clean.parquet")
corpus = pd.read_parquet(RESULTS_DIR / "01_user_corpus.parquet")

# Centralidad del notebook 02 (para contexto)
centrality = pd.read_parquet(RESULTS_DIR / "02_centrality.parquet")

print(f"Usuarios con corpus: {len(corpus):,}")
print(f"Columnas del corpus: {list(corpus.columns)}")
corpus.head(3)

## 2. Embeddings con qwen3-embedding

### ¿Qué es un embedding?

Un **embedding** es una representación numérica de un texto — un vector de números (en este caso, 4096 números) que captura el significado semántico del texto.

La propiedad más importante: **textos similares producen vectores similares**. Si dos usuarios escriben sobre los mismos temas con el mismo vocabulario, sus vectores estarán cerca en el espacio de 4096 dimensiones.

Esto nos permite:
- Medir similitud entre usuarios sin buscar palabras exactas
- Agrupar usuarios por similitud de contenido (clustering)
- Visualizar la distribución semántica del foro (UMAP)

**¿Por qué qwen3-embedding?**  
Es un modelo de embeddings de alto rendimiento que corre localmente vía Ollama. No necesitamos enviar datos a ninguna API externa — importante para datasets sensibles.

> **Nota de tiempo**: embedear todos los usuarios puede tomar 10-30 minutos dependiendo del hardware. El resultado se guarda en disco para no tener que repetirlo.

In [ ]:
EMBEDDINGS_PATH = RESULTS_DIR / "03_user_embeddings.npz"

if EMBEDDINGS_PATH.exists():
    # Cargar embeddings guardados (evita recalcular)
    data = np.load(EMBEDDINGS_PATH, allow_pickle=True)
    user_ids  = data["user_ids"]
    embeddings = data["embeddings"]
    print(f"Embeddings cargados desde disco: {len(user_ids):,} usuarios, dim={embeddings.shape[1]}")
else:
    # Generar embeddings con qwen3-embedding via Ollama
    # embed_users() toma la tabla de posts y genera un embedding por usuario
    # (concatena todos los posts del usuario y los embeda como un solo texto)
    from src.parsers.vbulletin import load_forum
    from src.utils import list_forums, merge_tables
    
    # Necesitamos los posts originales para embed_users
    posts = pd.read_parquet(RESULTS_DIR / "01_posts_clean.parquet")
    
    print("Generando embeddings (esto puede tardar varios minutos)...")
    user_ids, embeddings = embed_users(posts, min_posts=10)
    
    # Guardar para no repetir
    np.savez(EMBEDDINGS_PATH, user_ids=user_ids, embeddings=embeddings)
    print(f"Embeddings generados y guardados: {len(user_ids):,} usuarios, dim={embeddings.shape[1]}")

## 3. UMAP: visualización en 2D

### ¿Qué hace UMAP?

UMAP toma nuestros vectores de 4096 dimensiones (que no podemos visualizar directamente) y los proyecta en 2 dimensiones intentando que los puntos que eran cercanos en 4096D sigan siendo cercanos en 2D.

Es como aplanar un mapa del mundo en un papel: hay distorsión, pero la estructura general se preserva. Los clusters que vemos en 2D son reales — representan grupos de usuarios con contenido semánticamente similar.

**Parámetros importantes de UMAP:**
- `n_neighbors=15`: cuántos vecinos considera para construir la estructura local. Valores bajos → más detalles finos; valores altos → más estructura global.
- `min_dist=0.1`: qué tan apretados pueden estar los puntos en 2D. Valores pequeños → clusters más compactos.
- `metric="cosine"`: usamos similitud coseno (igual que los embeddings) en lugar de distancia euclidiana.

In [ ]:
import umap

print("Reduciendo dimensionalidad con UMAP (4096D → 2D)...")
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)
coords_2d = reducer.fit_transform(embeddings)

print(f"Reducción completada. Shape resultado: {coords_2d.shape}")

# Construir DataFrame para visualización
umap_df = pd.DataFrame({
    "userid": pd.to_numeric(user_ids, errors="coerce"),
    "umap_x": coords_2d[:, 0],
    "umap_y": coords_2d[:, 1],
})

# Enriquecer con metadata
umap_df = umap_df.merge(
    users[["userid", "forum", "username", "posts"]].drop_duplicates(subset="userid"),
    on="userid", how="left"
)

umap_df.head()

In [ ]:
# Visualización interactiva UMAP — coloreada por foro
fig = px.scatter(
    umap_df,
    x="umap_x", y="umap_y",
    color="forum",
    hover_data=["username", "posts"],
    title="Usuarios en espacio semántico — UMAP sobre qwen3-embedding",
    labels={"umap_x": "UMAP dim 1", "umap_y": "UMAP dim 2"},
    opacity=0.7,
    width=950, height=650
)
fig.update_traces(marker=dict(size=5))
fig.write_html(RESULTS_DIR / "03_umap_by_forum.html")
fig.show()

print("\nQué buscar en este gráfico:")
print("  - Clusters bien separados → grupos de usuarios con contenido muy distinto")
print("  - Mezcla de colores dentro de un cluster → el mismo tipo de contenido aparece en múltiples foros")
print("  - Puntos aislados → usuarios con un estilo o contenido muy diferente al resto")

## 4. HDBSCAN: clustering semántico

### ¿Qué es HDBSCAN?

HDBSCAN es un algoritmo de clustering que:
1. **No necesita que especifiquemos K** (el número de clusters). Los encuentra solo.
2. **Puede identificar puntos de ruido** (usuarios que no pertenecen a ningún cluster).
3. Es robusto a clusters de diferentes formas y densidades.

Lo aplicamos sobre las coordenadas UMAP (2D) para que sea más rápido, pero el resultado refleja la estructura semántica de los embeddings originales.

**`min_cluster_size`**: el tamaño mínimo de un cluster. Si es muy pequeño (5), detectamos muchos clusters pequeños. Si es grande (50), solo detectamos los grupos más prominentes.

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,    # Al menos 15 usuarios para formar un cluster
    min_samples=5,          # Densidad mínima local
    metric="euclidean",     # Sobre coordenadas UMAP 2D
    cluster_selection_method="eom"
)

cluster_labels = clusterer.fit_predict(coords_2d)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = (cluster_labels == -1).sum()

print(f"Clusters encontrados: {n_clusters}")
print(f"Puntos de ruido (no asignados a ningún cluster): {n_noise:,} ({100*n_noise/len(cluster_labels):.1f}%)")

umap_df["cluster"] = cluster_labels

# Distribución de tamaño de clusters
cluster_sizes = pd.Series(cluster_labels[cluster_labels >= 0]).value_counts().sort_values(ascending=False)
print(f"\nTamaño de clusters (top 10):")
print(cluster_sizes.head(10).to_string())

In [ ]:
# Visualización UMAP coloreada por cluster HDBSCAN
umap_plot = umap_df.copy()
umap_plot["cluster_str"] = umap_plot["cluster"].astype(str)
umap_plot.loc[umap_plot["cluster"] == -1, "cluster_str"] = "ruido"

fig = px.scatter(
    umap_plot,
    x="umap_x", y="umap_y",
    color="cluster_str",
    hover_data=["username", "forum", "posts"],
    title="Clusters semánticos de usuarios (HDBSCAN sobre UMAP)",
    labels={"cluster_str": "Cluster"},
    opacity=0.7,
    width=950, height=650
)
fig.update_traces(marker=dict(size=5))
fig.write_html(RESULTS_DIR / "03_umap_clusters.html")
fig.show()

## 5. BERTopic: ¿de qué habla el foro?

### ¿Qué es BERTopic?

BERTopic es una librería de modelado de tópicos que combina embeddings de lenguaje (como los que ya generamos) con clustering para extraer los **temas** principales de un corpus de texto.

La salida es una lista de tópicos, donde cada tópico está representado por sus palabras más características. Por ejemplo:
- Tópico 3: `dump, cvv, track2, fresh, bins, quality` → ventas de datos de tarjetas
- Tópico 7: `escrow, trusted, vouch, deal, btc` → sistema de confianza y pagos

**¿Por qué BERTopic y no LDA?**  
LDA (Latent Dirichlet Allocation) es el modelo clásico de tópicos, pero asume que los documentos son mezclas de tópicos con distribuciones estadísticas fijas. BERTopic usa embeddings semánticos, así que entiende que "cc" y "credit card" son lo mismo.

In [ ]:
from bertopic import BERTopic

# Usamos el corpus por usuario que construimos en el notebook 01
# Cada "documento" es el corpus completo de un usuario
docs = corpus["corpus"].tolist()
doc_user_ids = corpus["userid"].tolist()

print(f"Documentos para BERTopic: {len(docs):,}")
print(f"Longitud media (caracteres): {np.mean([len(d) for d in docs]):,.0f}")

# Configuramos BERTopic para usar los embeddings que ya calculamos
# (así no los recalcula — ahorra tiempo)
# Si los doc_user_ids no coinciden exactamente con user_ids, hay que alinear
topic_model = BERTopic(
    nr_topics="auto",       # Que BERTopic decida cuántos tópicos
    min_topic_size=10,      # Al menos 10 documentos por tópico
    verbose=True
)

# Si tenemos embeddings alineados con el corpus, los pasamos directamente
# De lo contrario, BERTopic generará sus propios embeddings (más lento)
try:
    # Alinear embeddings con el corpus
    uid_to_idx = {int(uid): i for i, uid in enumerate(user_ids)}
    corpus_embeddings = np.array([
        embeddings[uid_to_idx[int(uid)]] if int(uid) in uid_to_idx else np.zeros(embeddings.shape[1])
        for uid in doc_user_ids
    ])
    topics, probs = topic_model.fit_transform(docs, corpus_embeddings)
    print("BERTopic ejecutado con embeddings pre-calculados")
except Exception as e:
    print(f"Error con embeddings pre-calculados: {e}")
    print("Ejecutando BERTopic sin embeddings pre-calculados...")
    topics, probs = topic_model.fit_transform(docs)

print(f"\nTópicos encontrados: {len(set(topics)) - 1}")  # -1 por el tópico -1 (outliers)

In [ ]:
# Mostrar los tópicos más grandes con sus palabras características
topic_info = topic_model.get_topic_info()
print("Tópicos principales (excluyendo outliers):")
print(topic_info[topic_info["Topic"] >= 0].head(15)[["Topic", "Count", "Name", "Representation"]].to_string(index=False))

In [ ]:
# Visualización interactiva de BERTopic
fig_topics = topic_model.visualize_topics()
fig_topics.write_html(RESULTS_DIR / "03_bertopic_map.html")
fig_topics.show()

# Asignar tópico a cada usuario
corpus["topic"] = topics
corpus.to_parquet(RESULTS_DIR / "03_corpus_with_topics.parquet", index=False)

## 6. NER con qwen2.5:14b — extracción de entidades

### ¿Qué es NER?

**NER** (Named Entity Recognition) es la tarea de identificar y clasificar entidades específicas en un texto. En un contexto estándar, se buscan personas, organizaciones, lugares. En nuestro contexto de carding, nos interesan entidades muy específicas:

- **Herramientas**: software de hacking, checkers, skimmers
- **Productos**: dumps, CVVs, fullz, accounts
- **Métodos de pago**: Bitcoin, WMZ, Qiwi, PayPal
- **Mercados**: nombres de tiendas/servicios específicos
- **Marcas de tarjetas**: Visa, Mastercard, Amex (con características como BIN, país)

Usamos `qwen2.5:14b` — un LLM local de 14 mil millones de parámetros que entiende contexto de ciberseguridad.

**¿Por qué un LLM y no un NER clásico como spaCy?**  
El vocabulario del carding es muy específico y cambia rápido. Un modelo entrenado en texto general no va a reconocer que "fresh drops" se refiere a dumps recientes, o que "high balance" es un atributo de una tarjeta. El LLM lo infiere del contexto.

In [ ]:
import ollama

NER_PROMPT_TEMPLATE = """Eres un analista forense especializado en cibercrimen financiero.
Tu tarea es extraer entidades relevantes del siguiente texto de un foro de carding.

Categorías a extraer:
- TOOL: herramientas, software, checkers, skimmers
- PRODUCT: dumps, CVVs, fullz, accounts, logs
- PAYMENT: métodos de pago (Bitcoin, BTC, WMZ, Qiwi, etc.)
- MARKET: nombres de tiendas, servicios o mercados
- CARD_TYPE: tipos de tarjeta con atributos (Visa Platinum, Amex Gold, etc.)
- COUNTRY: países mencionados como origen de tarjetas o usuarios

Responde SOLO con un JSON válido en este formato:
{"entities": [{"text": "...", "type": "...", "count": 1}]}

Si no hay entidades relevantes, responde: {"entities": []}

Texto a analizar:
{text}"""

def extract_entities(text: str, max_chars: int = 2000) -> list:
    """Extrae entidades de un texto usando qwen2.5:14b vía Ollama."""
    # Truncar a max_chars para no exceder el contexto del modelo
    truncated = text[:max_chars] if len(text) > max_chars else text
    
    try:
        response = ollama.chat(
            model="qwen2.5:14b",
            messages=[{
                "role": "user",
                "content": NER_PROMPT_TEMPLATE.format(text=truncated)
            }],
            options={"temperature": 0}  # Determinista para NER
        )
        result = json.loads(response["message"]["content"])
        return result.get("entities", [])
    except json.JSONDecodeError:
        # El modelo a veces añade texto antes del JSON — intentamos extraerlo
        content = response["message"]["content"]
        start = content.find("{")
        if start >= 0:
            try:
                return json.loads(content[start:])["entities"]
            except:
                pass
        return []
    except Exception as e:
        return []

# Test rápido
test_text = "selling fresh dumps track1+track2, US banks only, BTC payment preferred, use escrow"
print("Test NER:")
print(f"  Texto: {test_text}")
print(f"  Entidades: {extract_entities(test_text)}")

In [ ]:
# Aplicar NER a los usuarios más activos (top 100 por posts)
# No procesamos todos los usuarios — es caro en tiempo y suficiente para el análisis
TOP_N_FOR_NER = 100

top_users_corpus = corpus.merge(
    users[["userid", "posts"]].drop_duplicates(subset="userid"),
    on="userid", how="left"
).nlargest(TOP_N_FOR_NER, "posts_y" if "posts_y" in corpus.columns else "posts")

NER_RESULTS_PATH = RESULTS_DIR / "03_ner_results.jsonl"

print(f"Aplicando NER a {len(top_users_corpus)} usuarios activos...")
print("(Cada usuario puede tardar 5-15 segundos)")

ner_results = []
for _, row in tqdm(top_users_corpus.iterrows(), total=len(top_users_corpus), desc="NER"):
    entities = extract_entities(row["corpus"])
    result = {
        "userid": row["userid"],
        "username": row.get("username", "unknown"),
        "forum": row.get("forum", "unknown"),
        "entities": entities
    }
    ner_results.append(result)

# Guardar como JSONL (una línea = un usuario)
with open(NER_RESULTS_PATH, "w") as f:
    for r in ner_results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"\nNER completado. Guardado en {NER_RESULTS_PATH}")

In [ ]:
# Análisis de las entidades más frecuentes
from collections import Counter

all_entities = []
for r in ner_results:
    for e in r["entities"]:
        all_entities.append((e.get("type", "UNK"), e.get("text", "").lower()))

entity_counts = Counter(all_entities)

# Mostrar top entidades por tipo
for entity_type in ["PRODUCT", "PAYMENT", "TOOL", "MARKET", "COUNTRY"]:
    type_entities = [(text, count) for (etype, text), count in entity_counts.most_common(200) if etype == entity_type]
    if type_entities:
        print(f"\nTop {entity_type}:")
        for text, count in type_entities[:10]:
            print(f"  {text}: {count}")

## 7. Perfilado de actores

Esta es la sección central del análisis semántico y la más novedosa: vamos a sintetizar un **perfil** de cada actor relevante a partir de las señales que hemos recolectado:

- **NER**: qué productos/herramientas/métodos menciona
- **Tópicos (BERTopic)**: a qué tema pertenece su contenido
- **Actividad**: cuántos posts, en qué subforos, en qué período
- **Centralidad de red**: degree y betweenness (del notebook 02)

Con todo esto, le pedimos a `qwen2.5:14b` que infiera el **rol** del usuario en el ecosistema de carding:
- Vendedor de dumps/CVVs
- Comprador / usuario final
- Checker (valida tarjetas)
- Intermediario de confianza / escrow
- Administrador / moderador
- Lurker / nuevo (poca señal)

In [ ]:
PROFILE_PROMPT_TEMPLATE = """Eres un analista forense especializado en cibercrimen financiero.
Analiza la información de este usuario de un foro de carding e infiere su rol.

INFORMACIÓN DEL USUARIO:
- Username: {username}
- Foro: {forum}
- Total de posts: {posts}
- Centralidad en la red (degree): {degree:.4f}
- Rol de puente en la red (betweenness): {betweenness:.4f}
- Cluster semántico: {cluster}
- Tópico principal: {topic}

ENTIDADES EXTRAÍDAS DE SUS POSTS:
{entities}

MUESTRA DE SUS POSTS (primeros 1500 caracteres):
{corpus_sample}

Basándote en toda esta información, responde con un JSON válido:
{{
  "role": "<vendedor_dumps|vendedor_cvv|comprador|checker|intermediario_escrow|administrador|lurker|indeterminado>",
  "confidence": <0.0-1.0>,
  "signals": ["señal 1", "señal 2", "señal 3"],
  "summary": "Una frase que describe al actor y su rol inferido",
  "caveats": "Limitaciones o incertidumbres en esta inferencia"
}}"""

def profile_actor(row, centrality_df, ner_results_by_user):
    """Genera un perfil del actor usando el LLM."""
    userid = row["userid"]
    
    # Recuperar centralidad
    cent = centrality_df[centrality_df["userid"] == userid]
    degree = float(cent["degree_centrality"].iloc[0]) if len(cent) else 0.0
    betweenness = float(cent["betweenness_centrality"].iloc[0]) if len(cent) else 0.0
    
    # Entidades NER
    ner_data = ner_results_by_user.get(userid, [])
    entities_str = ", ".join([f"{e.get('type')}: {e.get('text')}" for e in ner_data[:20]])
    
    prompt = PROFILE_PROMPT_TEMPLATE.format(
        username=row.get("username", "unknown"),
        forum=row.get("forum", "unknown"),
        posts=row.get("posts", 0),
        degree=degree,
        betweenness=betweenness,
        cluster=row.get("cluster", -1),
        topic=row.get("topic", -1),
        entities=entities_str if entities_str else "(ninguna extraída)",
        corpus_sample=str(row.get("corpus", ""))[:1500]
    )
    
    try:
        response = ollama.chat(
            model="qwen2.5:14b",
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.1}
        )
        content = response["message"]["content"]
        start = content.find("{")
        if start >= 0:
            return json.loads(content[start:])
    except Exception as e:
        pass
    
    return {"role": "indeterminado", "confidence": 0.0, "signals": [], "summary": "Error en el análisis", "caveats": str(e)}

print("Función de perfilado definida.")

In [ ]:
# Preparar los datos para el perfilado
# Enriquecer corpus con datos de centralidad y cluster
if "03_corpus_with_topics.parquet" in [p.name for p in RESULTS_DIR.iterdir()]:
    corpus_enriched = pd.read_parquet(RESULTS_DIR / "03_corpus_with_topics.parquet")
else:
    corpus_enriched = corpus.copy()
    corpus_enriched["topic"] = -1

corpus_enriched = corpus_enriched.merge(
    umap_df[["userid", "cluster"]],
    on="userid", how="left"
)

# Indexar NER por userid
ner_by_user = {r["userid"]: r["entities"] for r in ner_results}

# Seleccionar los top 30 usuarios más activos para perfilar
# (el perfilado completo lleva tiempo — hacemos los más relevantes)
TOP_TO_PROFILE = 30
top_to_profile = corpus_enriched.merge(
    users[["userid", "posts"]].drop_duplicates(subset="userid"),
    on="userid", how="left"
).nlargest(TOP_TO_PROFILE, "posts_y" if "posts_y" in corpus_enriched.columns else "posts")

print(f"Perfilando {len(top_to_profile)} actores...")

profiles = []
for _, row in tqdm(top_to_profile.iterrows(), total=len(top_to_profile), desc="Perfilando"):
    profile = profile_actor(row, centrality, ner_by_user)
    profile["userid"] = row["userid"]
    profile["username"] = row.get("username", "unknown")
    profile["forum"] = row.get("forum", "unknown")
    profiles.append(profile)

profiles_df = pd.DataFrame(profiles)
profiles_df.to_parquet(RESULTS_DIR / "03_actor_profiles.parquet", index=False)
print(f"\nPerfiles guardados en results/03_actor_profiles.parquet")

In [ ]:
# Tabla de actores con rol inferido
print("TABLA DE ACTORES CLAVE — ROLES INFERIDOS")
print("=" * 80)

display_cols = ["username", "forum", "role", "confidence", "summary"]
available_display = [c for c in display_cols if c in profiles_df.columns]

profiles_sorted = profiles_df.sort_values("confidence", ascending=False)
profiles_sorted[available_display]

In [ ]:
# Distribución de roles
if "role" in profiles_df.columns:
    role_dist = profiles_df["role"].value_counts()
    
    fig = px.bar(
        x=role_dist.index,
        y=role_dist.values,
        title="Distribución de roles inferidos (top 30 actores)",
        labels={"x": "Rol", "y": "Número de actores"},
        color=role_dist.values,
        color_continuous_scale="Viridis"
    )
    fig.write_html(RESULTS_DIR / "03_role_distribution.html")
    fig.show()

print("\n→ Siguiente paso: 04_sintesis_informe.ipynb")